# 09 - Live scoring of currently-confirmed bookings

Scores every booking that is **Confirmed right now** (open, not yet resolved) with
each trained model - LogReg (01), XGBoost (02), HistGB (03) and the hazard model (08) -
and writes an **Excel** file with the id, the relevant features and one
predicted-cancel-probability column per model, plus the scoring timestamp. Purpose:
park it, wait a few weeks, and manually check whether the flagged bookings actually
cancel. Requires the model notebooks (01/02/03/08) to have been run + persisted.

## 0 - Setup

In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here/"pyproject.toml").exists():
    if _here==_here.parent: raise RuntimeError("project root not found")
    _here=_here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from src import load_reservations, load_clean_reservations, color, data_dir, tables_dir
from src.features import load_feature_roster, family_feature_lists
import src.scoring as sc, src.hazard as HZ
pio.templates.default="plotly_white"
BRAND={n: color(n) for n in ["blue","orange","green","purple","red"]}
SCORED_AT = pd.Timestamp.now("UTC")
print("scoring timestamp (UTC):", SCORED_AT)

scoring timestamp (UTC): 2026-07-04 07:02:09.571535+00:00


## 1 - Load currently-confirmed bookings

`status == "Confirmed"` = open bookings that will resolve over the coming weeks.
`force_refresh=True` in the loader pulls fresh from BigQuery; here we read the cache.

In [2]:
raw = load_reservations()                     # add force_refresh=True for a live pull
conf = raw[raw["status"].astype("string") == "Confirmed"].copy()
if "id" not in conf.columns:
    conf["id"] = np.arange(len(conf))         # fallback key (raw usually carries `id`)
arr = pd.to_datetime(conf["arrival"], utc=True); cre = pd.to_datetime(conf["created"], utc=True)
conf["lead_days"] = (arr - cre) / pd.Timedelta(days=1)
conf["days_to_arrival"] = (arr - SCORED_AT) / pd.Timedelta(days=1)
print(f"confirmed bookings: {len(conf):,}")
print(f"  arrival range : {arr.min()} -> {arr.max()}")
print(f"  days-to-arrival: median {conf['days_to_arrival'].median():.0f}  "
      f"(within 14d: {(conf['days_to_arrival']<=14).sum():,})")
display(conf.groupby(conf['arrival'].dt.to_period('M').astype(str)).size()
            .rename('n_confirmed').to_frame().tail(12))

loading cached parquet: reservations_raw_no_pii.parquet


confirmed bookings: 5,130
  arrival range : 2026-06-30 13:00:00+00:00 -> 2027-12-26 14:00:00+00:00
  days-to-arrival: median 31  (within 14d: 1,777)


/var/folders/fg/gknf93rj4lxf9zgbszh30s6r0000gq/T/ipykernel_49643/3975424951.py:12: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  display(conf.groupby(conf['arrival'].dt.to_period('M').astype(str)).size()


,n_confirmed
arrival,
2026-09,802
2026-10,398
2026-11,205
2026-12,94
2027-01,73
2027-02,55
2027-03,35
2027-04,9
2027-05,7


## 2 - Build features (serving parity)

`src.scoring.build_features` reproduces 00's engineering (incl. the linear `_log`
twins) so every model family is scoreable; `apply_scoring_bounds` drops rows whose
essential features are impossible to compute (but never filters on status).

In [3]:
roster = load_feature_roster()
feat = sc.build_features(conf)                # dynamic features are point-in-time (now)
feat = sc.apply_scoring_bounds(feat)
print(f"scoreable bookings after bounds: {len(feat):,}")
need = roster["numeric"] + roster["categorical"]
missing = [c for c in need if c not in feat.columns]
assert not missing, f"build_features missing roster features: {missing}"
print("all roster features present ->", not missing)

scoreable bookings after bounds: 5,128
all roster features present -> True


## 3 - Score with every trained model

Each static model gets its family view (LogReg -> linear/log twins; trees -> raw);
the hazard model scores the survival product over each booking's remaining
days-until-arrival. Missing models are skipped with a warning.

In [4]:
out = feat.copy()
prob_cols = []
for m, family in [("logreg", "linear"), ("xgboost", "tree"), ("histgb", "tree")]:
    try:
        pipe = sc.load_model(m)
        num, cat = family_feature_lists(roster, family)
        out[f"p_{m}"] = pipe.predict_proba(out[num + cat])[:, 1]
        prob_cols.append(f"p_{m}")
        print(f"  scored {m}")
    except FileNotFoundError:
        print(f"  {m} not trained/persisted yet - skipped (run notebook first)")

try:
    hz = HZ.load_hazard()
    b = out.copy(); b["lead"] = b["lead_time_days"]
    b[HZ.AXIS] = b["days_until_arrival"].clip(lower=1)          # remaining days, live
    out["p_hazard"] = HZ.score_upcoming_hazard(hz, b)
    prob_cols.append("p_hazard")
    print("  scored hazard")
except FileNotFoundError:
    print("  hazard not trained/persisted yet - skipped")

assert prob_cols, "no models available - run 01/02/03/08 first"
if len(prob_cols) > 1:
    out["p_ensemble"] = out[prob_cols].mean(axis=1)            # simple mean of available models
    prob_cols_disp = prob_cols + ["p_ensemble"]
else:
    prob_cols_disp = prob_cols
print("probability columns:", prob_cols_disp)

  scored logreg
  scored xgboost
  scored histgb
  scored hazard
probability columns: ['p_logreg', 'p_xgboost', 'p_histgb', 'p_hazard', 'p_ensemble']


## 4 - Prediction summary

In [5]:
display(out[prob_cols_disp].describe().T.round(4))
fig = go.Figure()
for c in prob_cols_disp:
    fig.add_histogram(x=out[c], nbinsx=50, name=c, opacity=0.6)
fig.update_layout(barmode="overlay", title="Predicted cancel-probability distribution by model",
                  xaxis_title="P(cancel before arrival)", yaxis_title="bookings")
fig.show()

# agreement: pairwise correlation of the model probabilities
if len(prob_cols) > 1:
    display(out[prob_cols].corr().round(3))

,count,mean,std,min,25%,50%,75%,max
p_logreg,5128.0,0.2982,0.1229,0.0213,0.1900,0.3382,0.3717,0.5651
p_xgboost,5128.0,0.3124,0.0697,0.0016,0.2495,0.3361,0.3649,0.4169
p_histgb,5128.0,0.2954,0.0717,0.0000,0.2314,0.3170,0.3357,0.4156
p_hazard,5128.0,0.2307,0.2064,0.0000,0.0399,0.1908,0.3694,0.9938
p_ensemble,5128.0,0.2842,0.1079,0.0057,0.1814,0.2950,0.3631,0.5885


,p_logreg,p_xgboost,p_histgb,p_hazard
p_logreg,1.000,0.894,0.917,0.700
p_xgboost,0.894,1.000,0.971,0.719
p_histgb,0.917,0.971,1.000,0.715
p_hazard,0.700,0.719,0.715,1.000


## 5 - Excel export

id + key features + one probability column per model + the scoring timestamp, so you
can revisit in a few weeks and mark which bookings actually cancelled.

In [6]:
SCORED_AT_WRITE = pd.Timestamp.now()
KEEP = [c for c in ["id", "property_name", "channelCode", "guaranteeType",
                    "ratePlan_category", "arrival", "created", "lead_days",
                    "days_to_arrival", "gross_amount", "los_nights"] if c in out.columns]
export = out[KEEP + prob_cols_disp].copy()
export.insert(0, "scored_at_utc", SCORED_AT_WRITE)
export = export.sort_values(prob_cols_disp[-1], ascending=False)
export["arrival"] = export["arrival"].dt.tz_localize(None)
export["created"] = export["created"].dt.tz_localize(None)

XLSX = data_dir() / f"live_scores_{SCORED_AT_WRITE.strftime('%Y%m%d')}.xlsx"
export.to_excel(XLSX, index=False, sheet_name="live_scores")
print(f"exported {len(export):,} scored bookings -> {XLSX}")
display(export.head(20))

exported 5,128 scored bookings -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/Data/live_scores_20260704.xlsx


,scored_at_utc,id,property_name,channelCode,guaranteeType,ratePlan_category,arrival,created,lead_days,days_to_arrival,gross_amount,los_nights,p_logreg,p_xgboost,p_histgb,p_hazard,p_ensemble
195388,2026-07-04 09:02:13.995909,DRRGEUVS-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-09-19 13:00:00,2026-06-10 01:33:56,101.476435,77.248500,4968.60,84.0,0.565062,0.409914,0.404083,0.974762,0.588455
195170,2026-07-04 09:02:13.995909,COWOVEOK-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-10-03 13:00:00,2026-06-26 13:56:55,98.960475,91.248500,4554.55,77.0,0.559924,0.409914,0.404083,0.971978,0.586475
194848,2026-07-04 09:02:13.995909,XCNXCQHH-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-11-01 14:00:00,2026-02-20 15:39:34,253.930856,120.290167,1697.95,29.0,0.559924,0.409914,0.404083,0.955129,0.582262
167283,2026-07-04 09:02:13.995909,PCZJECFH-1,Berlin Friedrichshain,ChannelManager,CreditCard,flexible longstay,2026-07-10 13:00:00,2026-03-24 16:12:01,107.866655,6.248500,11630.13,87.0,0.503802,0.416852,0.407848,0.993818,0.580580
195169,2026-07-04 09:02:13.995909,YWXIHNBM-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-12-01 14:00:00,2026-06-25 14:41:48,158.970972,150.290167,1530.00,30.0,0.533294,0.409914,0.404083,0.949871,0.574290
167488,2026-07-04 09:02:13.995909,UGWQBFUA-1,Berlin Friedrichshain,ChannelManager,CreditCard,flexible longstay,2026-08-01 13:00:00,2026-04-27 13:37:07,95.974225,28.248500,7656.00,90.0,0.477678,0.409914,0.407848,0.992752,0.572048
195914,2026-07-04 09:02:13.995909,LQORGRYA-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible midstay,2026-09-03 13:00:00,2026-05-30 08:14:39,96.198160,61.248500,2953.34,27.0,0.559924,0.414742,0.415569,0.892064,0.570575
195002,2026-07-04 09:02:13.995909,DOPPNVRC-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-11-01 14:00:00,2026-06-16 09:29:11,138.188067,120.290167,1719.70,29.0,0.533294,0.409914,0.404083,0.932974,0.570066
195027,2026-07-04 09:02:13.995909,FWLEIOMM-1,Frankfurt Sachsenhausen,ChannelManager,CreditCard,flexible longstay,2026-10-01 13:00:00,2026-05-26 18:13:03,127.782604,89.248500,1779.00,30.0,0.533294,0.409914,0.404083,0.930848,0.569535
187628,2026-07-04 09:02:13.995909,NLJHRPSL-1,Cologne Ehrenfeld,ChannelManager,CreditCard,flexible longstay,2026-08-16 13:00:00,2026-01-24 23:04:32,203.580185,43.248500,5529.55,65.0,0.519983,0.400932,0.397976,0.923073,0.560491
